第103週 - タイムトラベルとクローン

使う機能：タイムトラベル機能

公式ドキュメント：https://docs.snowflake.com/ja/user-guide/data-time-travel#specifying-the-data-retention-period-for-an-object

＜できること＞

①特定の時間のデータを参照できる(1時間前とか時間を指定してそのタイミングのデータを見れるお！実質タイムマシン🤖)

②オブジェクトを元に戻せる(データを消したりする前のデータに戻せるお！精神的ダメージの軽減たすかる💝)

③ある時点の状態をクローンとして保存できる（過去のデータの状態のデータをコピー＝クローンとしてテーブル化できるお！テスト環境作る時とかに助かり申す🔁）




🌟今回の課題🌟

【１】3日前の状態に復元する：データ更新前(3日前)の状態に戻すンゴねぇ！

【２】昨日の朝の状態に復元する：データを消しちゃって詰んだ💦昨日の朝に戻せば元に戻せるンゴ！！

【３】6時間前の状態にロールバックする：間違えてデータ消しちゃったンゴねぇ。。。「6時間前」にロルバしてクレメンス！

【４】テスト環境のクローンを作成する：開発用に本番の過去時点のデータのクローンがほしいンゴ！

In [ ]:
%%sql -r dataframe_13
ALTER SESSION SET TIMEZONE = 'Asia/Tokyo';

SELECT CURRENT_TIMESTAMP()::TIMESTAMP_TZ;

In [ ]:
%%sql -r dataframe_4
CREATE ROLE IF NOT EXISTS FF_CLLENGE_ROLE;
GRANT ROLE FF_CLLENGE_ROLE TO ROLE SYSADMIN;


In [ ]:
%%sql -r dataframe_3
CREATE DATABASE IF NOT EXISTS FF_CLLENGE;
GRANT USAGE ON DATABASE FF_CLLENGE TO ROLE FF_CLLENGE_ROLE;

In [ ]:
%%sql -r dataframe_5
CREATE SCHEMA IF NOT EXISTS FF_CLLENGE.CLNG;

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE TABLE FF_CLLENGE.CLNG.transactions (
    id INT,
    customer_name STRING,
    amount DECIMAL(10,2),
    transaction_date TIMESTAMP
);



In [ ]:
%%sql -r dataframe_10
INSERT INTO FF_CLLENGE.CLNG.transactions (id, customer_name, amount, transaction_date) VALUES 
    (1, 'VAMPIRE', 100.00, '2025-06-20 10:00:00'), 
    (2, 'ANGEL', 200.00, '2025-06-20 11:00:00'), 
    (3, 'CEO', 300.00, '2025-06-20 12:00:00'),
    (4, 'HOST', 600.00, '2025-06-20 12:00:00'),
    (5, 'STUDENT', 400.00, '2025-06-20 12:00:00'),
    (6, 'SINGERSONGLIGHTER', 700.00, '2025-06-20 12:00:00');


In [ ]:
%%sql -r dataframe_2
--保持期間の確認(1:1日の意味)
show tables ;

In [ ]:
%%sql -r dataframe_11
--変更(３日前のデータが今回ほしいから、バッファで５日間)
ALTER TABLE FF_CLLENGE.CLNG.transactions SET DATA_RETENTION_TIME_IN_DAYS=5;

--保持期間の確認(1:1日の意味)
show tables ;
--データ確認
--select * from FF_CLLENGE.CLNG.transactions;

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

print("ACCOUNTADMINの権限があれば、保持期間は変更できる")
print("(DATA_RETENTION_TIME_IN_DAYS オブジェクトパラメーター)")
img = mpimg.imread("hozonkikan.png")
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
%%sql -r dataframe_6
--Issue 1 (three days ago):
--UPDATE FF_CLLENGE.CLNG.transactions SET amount = amount * 1.1 WHERE id = 1;
DELETE FROM FF_CLLENGE.CLNG.transactions WHERE id = 4;

--Issue 2 (Yesterday) :
--DELETE FROM FF_CLLENGE.CLNG.transactions WHERE id = 3;

--Issue 3 (6 hours ago):
--INSERT INTO FF_CLLENGE.CLNG.transactions (id, customer_name, amount, transaction_date)
--SELECT id, customer_name, amount, transaction_date FROM FF_CLLENGE.CLNG.transactions;


In [ ]:
%%sql -r dataframe_12
--ある特定のタイミングのデータの状況を取得
SELECT * FROM FF_CLLENGE.CLNG.transactions AT(TIMESTAMP => 'Thu, 11 Jun 2026 00:00:00 -0900'::timestamp_tz);
--SELECT * FROM FF_CLLENGE.CLNG.transactions


In [ ]:
%%sql -r dataframe_14
--秒単位での時間の計算が必要（５分前：-60*5、今22時なので朝８時は）
SELECT * FROM FF_CLLENGE.CLNG.transactions AT(OFFSET => -60*60*6);

In [ ]:
%%sql -r dataframe_15
--データを消してしまった6時間前に戻したい
--消した時のクエリIDを取得しておいて、その処理より前と指定する
SELECT * FROM FF_CLLENGE.CLNG.transactions BEFORE(STATEMENT => '01c4f8b0-0003-ef62-0004-c70e00059362');

In [ ]:
%%sql -r dataframe_16
--「transactions_before_clone」に「FF_CLLENGE.CLNG.transactions」のある時点のデータを取得してデータを作成

--CREATE TABLE FF_CLLENGE.CLNG.transactions_before_clone CLONE FF_CLLENGE.CLNG.transactions
--  BEFORE(STATEMENT => '01c4f8b0-0003-ef62-0004-c70e00059362');

  select * from FF_CLLENGE.CLNG.transactions_before_clone